In [34]:
import torchvision
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets,transforms
import matplotlib.pyplot as plt

In [35]:
class FashionMNIST_dataset:
    def __init__(self,batch_size=64):
        self.transform=transforms.Compose([transforms.ToTensor(),
                                          transforms.Normalize((0.5,),(0.5))])
        self.train_dataset=datasets.FashionMNIST(root='./data',train=True,transform=self.transform,download=True)
        self.test_dataset=datasets.FashionMNIST(root='./data',train=False,transform=self.transform,download=True)
        self.train_loader=DataLoader(dataset=self.train_dataset,batch_size=batch_size,shuffle=True)
        self.test_loader=DataLoader(dataset=self.test_dataset,batch_size=batch_size,shuffle=False)        

In [36]:
class cnn(nn.Module):
    def __init__(self):
        super(cnn,self).__init__()
        self.conv1=nn.Conv2d(1,32,kernel_size=3,padding=1)
        self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.pool=nn.MaxPool2d(kernel_size=2,stride=2)
        self.fc1=nn.Linear(64*7*7,128)
        self.fc2=nn.Linear(128,10)
        self.dropout=nn.Dropout(0.5)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=x.view(-1,64*7*7)
        x=F.relu(self.fc1(x))
        x=self.fc2(x)
        return x
    def fit(self,dataset,epochs,optimizer,loss):
        for epoch in range(epochs):
            self.train()
            running_loss=0.0
            for images , labels in dataset.train_loader:
                optimizer.zero_grad()
                outputs=self(images)
                loss=criterion(outputs,labels)
                loss.backward()
                optimizer.step()
                running_loss+=loss.item()
            print(f'epoch[{epoch+1}/{epoch}],loss:{running_loss/len(dataset.train_loader):.5f}')
    def evaluate(self,test_loader):
        self.eval()
        correct=0
        total=0
        with torch.no_grad():
            for images, labels in test_loader:
                outputs=self(images)
                _,predicted=torch.max(outputs.data,1)
                total+=labels.size(0)
                correct+=(predicted==labels).sum().item()
            print(f'test accuracy:{100* correct/total:.2f}%')

In [38]:
if __name__=='__main__':
    batch_size=64
    epochs=10
    learning_rate=0.001
    criterion=nn.CrossEntropyLoss()
    dataset=FashionMNIST_dataset(batch_size)
    model=cnn()
    loss=nn.CrossEntropyLoss()
    optimizer=optim.Adam(model.parameters(),learning_rate)
    model.fit(dataset,epochs,optimizer,loss)
    model.evaluate(dataset.test_loader)

epoch[1/0],loss:0.42884
epoch[2/1],loss:0.27188
epoch[3/2],loss:0.22240
epoch[4/3],loss:0.19130
epoch[5/4],loss:0.16638
epoch[6/5],loss:0.14412
epoch[7/6],loss:0.12387
epoch[8/7],loss:0.10560
epoch[9/8],loss:0.09035
epoch[10/9],loss:0.07547
test accuracy:91.69%
